In [1]:
import pandas as pd
import numpy as np

# ----------------------------
# CONDITION SELECTION
# Choose: "PUL" or "FLU"
# ----------------------------
COND = "FLU"   # change to "FLU" or "PUL" when needed

if COND not in {"PUL", "FLU"}:
    raise ValueError("COND must be either 'PUL' or 'FLU'")

# ----------------------------
# INPUT FILES
# ----------------------------
LIB_HIP_TSV = "./Input/YSC1055_HIP_all_genes.tsv"
LIB_HOP_TSV = "./Input/YSC1056_HOP_all_genes.tsv"

FLU_QTL_CSV = "./Input/Fluconazole_QTL_F_LOD05_20260804.csv"
PUL_QTL_CSV = "./Input/Pulvinatal_QTL_genes_20260330.csv"

print("COND:", COND)

COND: FLU


In [2]:
# ----------------------------
# USER-DEFINED THRESHOLDS
# ----------------------------
LOD_THRESHOLD   = 0.0
PLEIO_THRESHOLD = 0.0

# ----------------------------
# MODE SELECTION
#
# Mode 1: provide a list of genes
# Mode 2: All genes that pass threshold (**Set QUERY_GENES = None)
# ----------------------------

QUERY_GENES_FILE = "query_genes.txt"

# Mode 1: read query genes from file (one gene per line)
with open(QUERY_GENES_FILE, "r") as f:
    _genes = []
    for line in f:
        s = line.strip()
        if not s or s.startswith("#"):
            continue
        _genes.append(s.upper())

# optional: de-duplicate while preserving order
seen = set()
QUERY_GENES = [g for g in _genes if not (g in seen or seen.add(g))]

# Uncomment for Mode 2:
# QUERY_GENES = None

print("LOD_THRESHOLD:", LOD_THRESHOLD)
print("PLEIO_THRESHOLD:", PLEIO_THRESHOLD)
print("QUERY_GENES:", QUERY_GENES)

LOD_THRESHOLD: 0.0
PLEIO_THRESHOLD: 0.0
QUERY_GENES: ['MTR3', 'NSR1', 'RTS3', 'YGR161W-C', 'TIF4631', 'MRPS35', 'TRS65']


In [3]:
def load_library(tsv_path: str, library_name: str) -> pd.DataFrame:
    df = pd.read_csv(tsv_path, sep="\t")

    required = {"standard_name", "orf", "plate", "row", "col"}
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{library_name}: missing columns {missing}")

    df = df.copy()
    df["library"] = library_name

    # normalize
    df["standard_name"] = df["standard_name"].astype(str).str.strip()
    df["orf"] = df["orf"].astype(str).str.strip()
    df["row"] = df["row"].astype(str).str.strip()
    df["col"] = pd.to_numeric(df["col"], errors="coerce").astype("Int64")
    df["plate"] = pd.to_numeric(df["plate"], errors="coerce").astype("Int64")

    return df


lib_hip = load_library(LIB_HIP_TSV, "YSC1055_HIP")
lib_hop = load_library(LIB_HOP_TSV, "YSC1056_HOP")

lib_all = pd.concat([lib_hip, lib_hop], ignore_index=True)

# Build lookup allowing standard_name OR ORF
lookup_rows = []
for _, r in lib_all.iterrows():
    lookup_rows.append((r["standard_name"], r["library"], r["plate"], r["row"], r["col"], r["standard_name"], r["orf"]))
    lookup_rows.append((r["orf"],          r["library"], r["plate"], r["row"], r["col"], r["standard_name"], r["orf"]))

lib_lookup = pd.DataFrame(
    lookup_rows,
    columns=["gene_key", "library", "plate", "row", "col", "standard_name", "orf"]
)

lib_lookup["gene_key"] = lib_lookup["gene_key"].astype(str).str.strip()

print("Library lookup ready.")
print("Unique gene keys:", lib_lookup["gene_key"].nunique())

Library lookup ready.
Unique gene keys: 11284


In [4]:
def load_qtl(csv_path: str, qtl_name: str) -> pd.DataFrame:
    df = pd.read_csv(csv_path)

    required = {
        "interval_id", "chrom", "start", "end",
        "genes", "genes_std", "pleio_score", "lod_max"
    }
    missing = required - set(df.columns)
    if missing:
        raise ValueError(f"{qtl_name}: missing columns {missing}")

    df = df.copy()
    df["qtl_source"] = qtl_name

    df["lod_max"] = pd.to_numeric(df["lod_max"], errors="coerce")
    df["pleio_score"] = pd.to_numeric(df["pleio_score"], errors="coerce")
    df["chrom"] = pd.to_numeric(df["chrom"], errors="coerce").astype("Int64")
    df["start"] = pd.to_numeric(df["start"], errors="coerce").astype("Int64")
    df["end"] = pd.to_numeric(df["end"], errors="coerce").astype("Int64")

    return df


def split_gene_list(x):
    if pd.isna(x):
        return []
    return [g.strip() for g in str(x).split(",") if g.strip()]


flu_qtl = load_qtl(FLU_QTL_CSV, "Fluconazole")
pul_qtl = load_qtl(PUL_QTL_CSV, "Pulvinatal")

# ----------------------------
# Select QTL source based on COND
# ----------------------------
if COND == "FLU":
    qtl_all = flu_qtl.copy()
elif COND == "PUL":
    qtl_all = pul_qtl.copy()

print("Using QTL condition:", COND)
print("QTL rows loaded:", len(qtl_all))

rows = []
for _, r in qtl_all.iterrows():
    for g in split_gene_list(r["genes_std"]):
        rows.append((g, "standard_name", r["qtl_source"], r["interval_id"],
                     r["chrom"], r["start"], r["end"], r["lod_max"], r["pleio_score"]))
    for g in split_gene_list(r["genes"]):
        rows.append((g, "orf", r["qtl_source"], r["interval_id"],
                     r["chrom"], r["start"], r["end"], r["lod_max"], r["pleio_score"]))

qtl_genes_long = pd.DataFrame(
    rows,
    columns=[
        "gene_key", "gene_key_type", "qtl_source", "interval_id",
        "chrom", "start", "end", "lod_max", "pleio_score"
    ]
)

qtl_genes_long["gene_key"] = qtl_genes_long["gene_key"].astype(str).str.strip()

print("QTL gene table ready.")
print("Rows:", len(qtl_genes_long))

Using QTL condition: FLU
QTL rows loaded: 33
QTL gene table ready.
Rows: 1668


In [5]:
def resolve_plate_locations(
    qtl_gene_table: pd.DataFrame,
    lib_lookup: pd.DataFrame,
    lod_threshold: float,
    pleio_threshold: float,
    query_genes: list[str] | None = None
) -> pd.DataFrame:
    """
    If query_genes is None:
        → return ALL genes passing thresholds
    If query_genes is a list:
        → restrict to those genes only

    Final output keeps unique gene-library-plate-row-col pairs.
    This removes duplicate QTL/SNP hits for the same gene-location,
    while retaining genes that are present in multiple wells/plates.
    """

    if query_genes is not None and len(query_genes) == 0:
        raise ValueError("query_genes is empty. Use None to select all genes.")

    # Filter by thresholds
    qtl_pass = qtl_gene_table[
        (qtl_gene_table["lod_max"] >= lod_threshold) &
        (qtl_gene_table["pleio_score"] >= pleio_threshold)
    ].copy()

    # Restrict to query genes if provided
    if query_genes is not None:
        query_genes_clean = [g.strip() for g in query_genes]
        qtl_pass = qtl_pass[qtl_pass["gene_key"].isin(query_genes_clean)]

    # Attach plate locations
    out = qtl_pass.merge(lib_lookup, how="left", on="gene_key")

    # Keep unique gene-plate/well pairs
    # Retains same gene if it occurs in multiple libraries, plates, rows, or columns
    out = (
        out.sort_values(
            ["gene_key", "library", "plate", "row", "col", "lod_max", "pleio_score"],
            ascending=[True, True, True, True, True, False, False],
            na_position="last"
        )
        .drop_duplicates(
            subset=["gene_key", "library", "plate", "row", "col"],
            keep="first"
        )
    )

    out = out[[
        "gene_key", "gene_key_type",
        "qtl_source", "interval_id",
        "chrom", "start", "end",
        "lod_max", "pleio_score",
        "library", "plate", "row", "col",
        "standard_name", "orf"
    ]].sort_values(
        ["gene_key", "library", "plate", "row", "col"],
        na_position="last"
    )

    return out

## **Mode 1: Query genes**

In [6]:
results = resolve_plate_locations(
    qtl_gene_table=qtl_genes_long,
    lib_lookup=lib_lookup,
    lod_threshold=LOD_THRESHOLD,
    pleio_threshold=PLEIO_THRESHOLD,
    query_genes=QUERY_GENES
)

# Reorder columns for human-friendly plate usage
results_table = results[[
    "gene_key",
    "standard_name",
    "orf",
    "gene_key_type",
    "qtl_source",
    "interval_id",
    "chrom",
    "start",
    "end",
    "lod_max",
    "pleio_score",
    "library",
    "plate",
    "row",
    "col"
]].copy()

print("Number of unique gene-plate/well rows:", len(results_table))
print("Number of unique genes:", results_table["gene_key"].nunique())

results_table

Number of unique gene-plate/well rows: 14
Number of unique genes: 7


,gene_key,standard_name,orf,gene_key_type,qtl_source,interval_id,chrom,start,end,lod_max,pleio_score,library,plate,row,col
13,MRPS35,MRPS35,YGR165W,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1055_HIP,216,C,7
14,MRPS35,MRPS35,YGR165W,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1056_HOP,36,E,11
15,MRPS35,MRPS35,YGR165W,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1056_HOP,67,E,3
0,MTR3,MTR3,YGR158C,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1055_HIP,216,C,1
1,NSR1,NSR1,YGR159C,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1055_HIP,216,C,2
2,NSR1,NSR1,YGR159C,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1056_HOP,36,E,6
5,RTS3,RTS3,YGR161C,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1055_HIP,216,C,4
6,RTS3,RTS3,YGR161C,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1056_HOP,36,E,8
11,TIF4631,TIF4631,YGR162W,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1055_HIP,285,h,10
12,TIF4631,TIF4631,YGR162W,standard_name,Fluconazole,15200,7,806826,829946,12.079738,0.023203,YSC1056_HOP,60,C,3


## **Optional Export**

In [7]:
SAVE_TSV = True        # set to False to disable
OUT_TSV  = "Output/FLU_F_LOD05_CHR7_TIF4631.tsv"

if SAVE_TSV:
    results_table.to_csv(OUT_TSV, sep="\t", index=False)
    print(f"Saved TSV to: {OUT_TSV}")
else:
    print("SAVE_TSV=False → not writing file.")

Saved TSV to: Output/FLU_F_LOD05_CHR7_TIF4631.tsv


# No. of Unique plates given the query genes

In [8]:
# ----------------------------
# Count unique plates from any exported results TSV
# ----------------------------

import pandas as pd

# Read the same file you saved earlier
df_out = pd.read_csv(OUT_TSV, sep="\t")

# Ensure plate column exists
if "plate" not in df_out.columns:
    raise ValueError(f"'plate' column not found in {OUT_TSV}. Columns: {list(df_out.columns)}")

# Count unique plates (ignore NaNs)
n_unique_plates = df_out["plate"].dropna().nunique()

print(f"File: {OUT_TSV}")
print(f"Unique plates: {n_unique_plates}")

# Optional: show the plate IDs
unique_plates = sorted(df_out["plate"].dropna().unique())
print("Plate IDs:", unique_plates)

File: Output/PUL_F_LOD10_Top10-Previous_location.tsv
Unique plates: 30
Plate IDs: [3.0, 5.0, 8.0, 16.0, 30.0, 54.0, 55.0, 59.0, 60.0, 63.0, 66.0, 68.0, 69.0, 70.0, 71.0, 72.0, 204.0, 212.0, 232.0, 237.0, 242.0, 255.0, 256.0, 257.0, 261.0, 281.0, 282.0, 283.0, 284.0, 285.0]


# No. of common plates given the query genes

In [18]:
# ----------------------------
# Within EACH library: plates shared between multiple genes + which genes
# ----------------------------

import pandas as pd

df_out = pd.read_csv(OUT_TSV, sep="\t")

required_cols = {"library", "plate", "standard_name"}
missing = required_cols - set(df_out.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}")

df = df_out.dropna(subset=["plate", "standard_name", "library"]).copy()

# Normalize types
df["plate"] = df["plate"].astype(str)
df["standard_name"] = df["standard_name"].astype(str)

for lib, dflib in df.groupby("library", sort=False):
    # genes per plate (within this library)
    genes_per_plate = (
        dflib.groupby("plate")["standard_name"]
        .nunique()
        .sort_values(ascending=False)
    )

    # plates that are shared by >=2 genes (within this library)
    shared_plates = genes_per_plate[genes_per_plate >= 2].index.tolist()

    print("=" * 60)
    print(f"Library: {lib}")
    print(f"Plates with >=2 genes (shared plates): {len(shared_plates)}")

    if len(shared_plates) == 0:
        print("No shared plates (each plate has genes from only one query gene).")
        continue

    # list genes on each shared plate
    plate_to_genes = (
        dflib[dflib["plate"].isin(shared_plates)]
        .groupby("plate")["standard_name"]
        .apply(lambda x: sorted(set(x)))
        .sort_index()
    )

    print("\nShared plates and the genes on them:")
    for plate, genes in plate_to_genes.items():
        print(f"Plate {plate}: {genes}")

    # Optional: a tidy table you can view/save
    shared_table = (
        dflib[dflib["plate"].isin(shared_plates)]
        .loc[:, ["library", "plate", "standard_name", "orf", "row", "col"]]
        .drop_duplicates()
        .sort_values(["plate", "standard_name"])
    )

    display(shared_table)

Library: YSC1055_HIP
Plates with >=2 genes (shared plates): 43

Shared plates and the genes on them:
Plate 201.0: ['AIM2', 'BDH1', 'BDH2', 'CNE1', 'ECM1', 'FLC2', 'GDH3', 'GEM1', 'GPB2', 'OAF1', 'PEX22']
Plate 204.0: ['ERG6', 'ERV25', 'GIS4', 'GLO1', 'MRPL39', 'PPZ1', 'PSP2', 'RAD33', 'SPT5', 'TAF11', 'TIF11', 'TRM12', 'TRM732', 'TRM9', 'UBX2', 'YAP1', 'YML003W', 'YML018C', 'YMR262W']
Plate 206.0: ['PPA2', 'RRN9', 'SCS7', 'ZDS1']
Plate 207.0: ['AKR2', 'ALG8', 'ASE1', 'CKA2', 'CUE5', 'CYC2', 'CYT1', 'DFG16', 'ETT1', 'EXO1', 'GLO4', 'GYP1', 'HIR2', 'HMS1', 'IRC23', 'LPL1', 'MSA1', 'PEP12', 'RSB1', 'SHE4', 'STD1', 'TMC1', 'TOM6', 'VHS3', 'VPS5', 'WHI2', 'YNG1', 'YOR041C', 'YOR050C', 'YOR053W', 'YOR055W', 'YOR062C']
Plate 212.0: ['AIM7', 'CDC34', 'DBF4', 'DET1', 'DOA4', 'DOS2', 'FMP16', 'IPT1', 'LCB2', 'MAK21', 'OCA6', 'PST1', 'RPS13', 'RRG1', 'RTR2', 'SNF11', 'TPI1', 'UBC5', 'VMS1', 'YDR061W', 'YOS9']
Plate 215.0: ['ARB1', 'CHZ1', 'EDC2', 'FIR1', 'GAL83', 'GCD11', 'GPA2', 'PHM8', 'PRO3', 

,library,plate,standard_name,orf,row,col
12,YSC1055_HIP,201.0,AIM2,YAL049C,B,3.0
75,YSC1055_HIP,201.0,BDH1,YAL060W,A,8.0
79,YSC1055_HIP,201.0,BDH2,YAL061W,A,7.0
132,YSC1055_HIP,201.0,CNE1,YAL058W,A,10.0
199,YSC1055_HIP,201.0,ECM1,YAL059W,A,9.0
...,...,...,...,...,...,...
722,YSC1055_HIP,285.0,SAM2,YDR502C,a,4.0
728,YSC1055_HIP,285.0,SEC20,YDR498C,d,2.0
833,YSC1055_HIP,285.0,TIF4631,YGR162W,h,10.0
945,YSC1055_HIP,285.0,WWM1,YFL010C,b,4.0


Library: YSC1056_HOP
Plates with >=2 genes (shared plates): 38

Shared plates and the genes on them:
Plate 1.0: ['ARD1', 'ARG4', 'DIA4', 'LAG1', 'MIP6', 'QCR10', 'RPL27A', 'SHU1', 'SPO13', 'STE20', 'VPS29', 'YHL005C']
Plate 10.0: ['AIM23', 'DAS1', 'IDS2', 'IRC9', 'LCB3', 'MRS3', 'RPA34', 'SMT1', 'URA2', 'YJL132W', 'YJL135W']
Plate 14.0: ['ACF2', 'APS1', 'ATG26', 'COQ9', 'DPH5', 'GID11', 'HMX1', 'IDP2', 'MDL1', 'MMR1', 'PBA1', 'PCD1', 'PEX13', 'RFX1', 'RPL37A', 'SAM1', 'SKG3', 'SWI6', 'TAG1', 'TFS1', 'TOS4', 'UPS2', 'VTA1', 'YKE2', 'YLR152C', 'YLR169W', 'YLR171W', 'YLR177W', 'YLR179C', 'YLR202C']
Plate 15.0: ['CHS5', 'FKS1', 'JIP3', 'MID2', 'NUP2', 'OPI9', 'RPL26A', 'RPS25B', 'SPO77', 'VRP1', 'YLR345W']
Plate 16.0: ['ERG6', 'ERV25', 'GIS4', 'GLO1', 'MRPL39', 'PPZ1', 'PSP2', 'RAD33', 'TRM12', 'TRM9', 'UBX2', 'YAP1', 'YML003W', 'YML018C']
Plate 18.0: ['PPA2', 'ROY1', 'RSN1', 'SCS7', 'TRM732', 'YMR262W', 'ZDS1']
Plate 20.0: ['AFI1', 'AKR2', 'ALG8', 'ASE1', 'CKA2', 'CUE5', 'CYC2', 'CYT1', '

,library,plate,standard_name,orf,row,col
49,YSC1056_HOP,1.0,ARD1,YHR013C,D,8.0
52,YSC1056_HOP,1.0,ARG4,YHR018C,D,11.0
184,YSC1056_HOP,1.0,DIA4,YHR011W,D,6.0
381,YSC1056_HOP,1.0,LAG1,YHL003C,D,3.0
439,YSC1056_HOP,1.0,MIP6,YHR015W,D,10.0
...,...,...,...,...,...,...
267,YSC1056_HOP,9.0,GDH3,YAL062W,C,4.0
276,YSC1056_HOP,9.0,GEM1,YAL048C,D,2.0
303,YSC1056_HOP,9.0,GPB2,YAL056W,C,9.0
521,YSC1056_HOP,9.0,OAF1,YAL051W,C,12.0


## Debug why a query gene failed

In [ ]:
def summarize_query_gene_thresholds(qtl_gene_table, query_genes):
    q = qtl_gene_table[qtl_gene_table["gene_key"].isin(query_genes)]
    return (
        q.groupby(["gene_key", "qtl_source"], as_index=False)
         .agg(
             best_lod=("lod_max", "max"),
             best_pleio=("pleio_score", "max"),
             n_intervals=("interval_id", "nunique")
         )
         .sort_values(["gene_key", "qtl_source"])
    )


if QUERY_GENES is not None:
    summarize_query_gene_thresholds(qtl_genes_long, QUERY_GENES)